In [1]:
print("hello, this is my capstone")


hello, this is my capstone


# Phase 2: Data Collection and Pipeline
## London Airbnb Analysis — MOHAN RAJU KOLIKONDA

**Data source:** Inside Airbnb (insideairbnb.com), London snapshot, June 2026
**Collection method:** Direct download of listings.csv.gz and reviews.csv.gz

In [2]:
import os
print("This notebook is running from:", os.getcwd())
print()
print("Files here:", os.listdir('.'))
print()
print("Does ../data/raw exist?", os.path.exists('../data/raw'))
if os.path.exists('../data/raw'):
    print("Files in ../data/raw:", os.listdir('../data/raw'))

This notebook is running from: C:\Users\K Mohan Raju\Untitled Folder 1\london-airbnb-analysis\notebooks

Files here: ['.ipynb_checkpoints', '01_pipeline.ipynb']

Does ../data/raw exist? True
Files in ../data/raw: ['listings.csv.gz', 'reviews.csv.gz']


In [3]:
import pandas as pd

listings = pd.read_csv('../data/raw/listings.csv.gz',
                       compression='gzip', low_memory=False)
reviews = pd.read_csv('../data/raw/reviews.csv.gz',
                      compression='gzip', parse_dates=['date'])

In [4]:
print(f"Listings: {listings.shape[0]:,} rows * {listings.shape[1]} columns")
print(f"Reviews:  {reviews.shape[0]:,} rows * {reviews.shape[1]} columns")

Listings: 92,638 rows * 90 columns
Reviews:  2,241,353 rows * 6 columns


## Dataset Requirments verification 
The capstone requires the datasets to meet seven minimum standards.
Each is verified with code below

In [8]:
rows = listings.shape[0]
print(f"Requirement 1 - At least 2000 rows: {rows:,} rows -> {'PASS' if rows >= 2000 else 'FAIL'}")

Requirement 1 - At least 2000 rows: 92,638 rows -> PASS


In [11]:
print("Earliest review:", reviews['date'].min())
print("Latest review:  ",reviews['date'].max())
months = (reviews['date'].max() - reviews['date'].min()).days / 30
print(f"Requirement 2 - 12+ months of dates: ~{months:.0f} months -> {'PASS' if months >= 12 else 'FAIL'}")       

Earliest review: 2009-12-21 00:00:00
Latest review:   2026-06-30 00:00:00
Requirement 2 - 12+ months of dates: ~201 months -> PASS


In [13]:
numeric_cols = listings.select_dtypes('number').columns
print(f"Requirement 3 - 3+numeric columns: {len(numeric_cols)} found -> {'PASS' if len(numeric_cols) >= 3 else 'FAIL'}")
print("Examples:", list(numeric_cols[:8]))

Requirement 3 - 3+numeric columns: 62 found -> PASS
Examples: ['id', 'scrape_id', 'neighborhood_overview', 'host_id', 'host_profile_id', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months']


In [14]:
for col in ['room_type', 'neighbourhood_cleansed', 'property_type']:
    print(f"{col}: {listings[col].nunique()}distinct values")
print("Requirement 4 - 3 categorical columns, one with 3+ values -> PASS"
      if listings['room_type'].nunique() >= 3 else "FAIL")

room_type: 4distinct values
neighbourhood_cleansed: 33distinct values
property_type: 91distinct values
Requirement 4 - 3 categorical columns, one with 3+ values -> PASS


In [15]:
text_entries = reviews['comments'].notna().sum()
print(f"Requirement 5 - text column with 200+ entries: {text_entries:,} review comments -> {'PASS' if text_entries >= 200 else 'FAIL'}")

Requirement 5 - text column with 200+ entries: 2,241,112 review comments -> PASS


In [16]:
match_rate = reviews['listing_id'].isin(listings['id']).mean() * 100
print(f"Requirement 6 - joinable tables: {match_rate:.1f}% of reviews match a listing id -> {'PASS' if match_rate > 0 else 'FAIL'}")

Requirement 6 - joinable tables: 99.8% of reviews match a listing id -> PASS


In [17]:
print(listings['host_is_superhost'].value_counts(dropna=False))
print("Requirement 7 - ML target variable available -> PASS")

host_is_superhost
f      75003
t      17559
NaN       76
Name: count, dtype: int64
Requirement 7 - ML target variable available -> PASS


## Data Dictionary
The listings table has 90 columns. Documenting all 90 adds noise, not value, so this dictionary covers the columns selected for analysis, plus the reasons for excluding the rest.

In [18]:
print(listings.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_nig

### Excluded columns
- **URLs and images** (listing_url, picture_url, host_url, host_thumbnail_url...): no analytical value
- **Scrape metadata** (scrape_id, last_scraped, calendar_last_scraped...): describes the collection, not the listings
- **Redundant night variants** (minimum_minimum_nights, maximum_minimum_nights...): near-duplicates of minimum_nights
- **Free-text about hosts** (host_about, neighborhood_overview): not needed; review comments are the NLP text source

In [19]:
keep_cols = [
    # identifiers and joins
    'id', 'host_id',
    # listing characteristics
    'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text',
    # location
    'neighbourhood_cleansed', 'latitude', 'longitude',
    # pricing
    'price',
    # host attributes
    'host_since', 'host_is_superhost', 'host_listings_count',
    # availability and activity
    'minimum_nights', 'availability_365', 'number_of_reviews', 'reviews_per_month',
    'first_review', 'last_review', 'estimated_occupancy_l365d', 'estimated_revenue_l365d',
    # review scores
    'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location', 'review_scores_value',
    # booking settings
    'instant_bookable'
]
print(len(keep_cols), "columns selected")
listings_selected = listings[keep_cols]

28 columns selected


In [20]:
summary = pd.DataFrame({
    'dtype': listings_selected.dtypes.astype(str),
    'pct_missing': (listings_selected.isna().mean() * 100).round(1),
    'n_unique': listings_selected.nunique(),
    'example': listings_selected.iloc[0]
})
summary

,dtype,pct_missing,n_unique,example
id,int64,0.0,92638,11551
host_id,int64,0.0,52125,43039
property_type,object,0.0,91,Entire rental unit
room_type,object,0.0,4,Entire home/apt
accommodates,int64,0.0,16,5
bedrooms,float64,25.9,21,1.0
beds,float64,35.1,28,3.0
bathrooms_text,object,0.1,49,1 bath
neighbourhood_cleansed,object,0.0,33,Lambeth
latitude,float64,0.0,53890,51.46095


| Column | Type | Meaning | Known quality issues |
|---|---|---|---|
| id | int | Unique listing identifier | None |
| price | text | Nightly price as displayed | Stored as text with "$" and commas — needs conversion; X% missing |
| host_is_superhost | text | Whether host has superhost status | Coded "t"/"f" not True/False; X% missing |
| review_scores_rating | float | Guest rating 0–5 | Missing for never-reviewed listings (X%) |

In [21]:
summary = pd.DataFrame({
    'dtype': listings_selected.dtypes.astype(str),
    'pct_missing': (listings_selected.isna().mean() * 100).round(1),
    'n_unique': listings_selected.nunique(),
    'example': listings_selected.iloc[0]
})
summary

,dtype,pct_missing,n_unique,example
id,int64,0.0,92638,11551
host_id,int64,0.0,52125,43039
property_type,object,0.0,91,Entire rental unit
room_type,object,0.0,4,Entire home/apt
accommodates,int64,0.0,16,5
bedrooms,float64,25.9,21,1.0
beds,float64,35.1,28,3.0
bathrooms_text,object,0.1,49,1 bath
neighbourhood_cleansed,object,0.0,33,Lambeth
latitude,float64,0.0,53890,51.46095
